# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step exploration of the FAIR^2 dataset using the `mlcroissant` library. The dataset includes clinicopathological and molecular features of second primary colorectal cancer in cancer survivors, as defined by its Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata as an object (not as a dict or list)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Number of records sets: {len(dataset.metadata.record_sets)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will iterate through the record sets, printing their `@id` and listing contained fields and columns. All references use the `@id` for clarity and reproducibility.

In [ ]:
# List record sets and their fields
record_sets = dataset.metadata.record_sets
print("Available record sets:")
for rs in record_sets:
    print(f"- RecordSet name: {rs.name} | @id: {rs.id}")
    fields = rs.fields
    print("  Fields:")
    for f in fields:
        print(f"    - Field name: {f.name}, @id: {f.id}, dataType: {f.data_type}")
    if hasattr(rs, 'columns') and rs.columns:
        print("  Columns:")
        for col in rs.columns:
            print(f"    - Column name: {col.name}, @id: {col.id}")
    print()

# Select the first record set's @id for previewing records
if record_sets:
    first_rs_id = record_sets[0].id
    print(f"Previewing records for RecordSet @id: {first_rs_id}")
    for idx, x in enumerate(dataset.records(record_set=first_rs_id)):
        print(x)
        if idx >= 2:
            break

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s discovered above.

We dynamically extract all record sets available in the dataset.

In [ ]:
# Extract data from all record sets (using @id)
dataframes = {}
rs_ids = [rs.id for rs in dataset.metadata.record_sets]

for rs_id in rs_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"RecordSet @id: {rs_id} -> Record count: {len(records)}")
        print(f"Columns: {dataframes[rs_id].columns.tolist()}")
        print(dataframes[rs_id].head(), '\n')
    else:
        print(f"RecordSet @id: {rs_id} has no records.")

# For demonstration, pick the first populated record set
primary_rs_id = next((rsid for rsid in rs_ids if rsid in dataframes), None)
if primary_rs_id:
    print(f"Using RecordSet @id: {primary_rs_id} for further analysis.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalization, and grouping.

We select a numeric field and a grouping field by their `@id` for analysis.

In [ ]:
# EDA using fields' @id from schema
import numpy as np
df = dataframes[primary_rs_id]
rs_obj = next((rs for rs in dataset.metadata.record_sets if rs.id == primary_rs_id), None)

# Choose first numeric field for demo
numeric_field_id = None
for f in rs_obj.fields:
    if f.data_type in ['Integer', 'Float', 'Number'] and f.id in df.columns:
        numeric_field_id = f.id
        break

# Choose a group field (prefer a non-numeric, categorical field)
group_field_id = None
for f in rs_obj.fields:
    if f.data_type == 'Text' and f.id in df.columns:
        group_field_id = f.id
        break

if numeric_field_id:
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize numeric field
    mean_val = filtered_df[numeric_field_id].mean()
    std_val = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean_val) / std_val
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric fields available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We plot the distribution of the selected numeric field and a bar plot by group.

In [ ]:
import matplotlib.pyplot as plt

if numeric_field_id:
    plt.figure(figsize=(8,4))
    df[numeric_field_id].hist(bins=15)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id:
        grouped = df.groupby(group_field_id)[numeric_field_id].mean()
        grouped.plot(kind='bar', figsize=(8,4))
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()
else:
    print("No numeric fields available for visualization.")

## 6. Conclusion

- We successfully loaded and explored the FAIR^2 dataset using its Croissant schema via `mlcroissant`, referencing all entities by their `@id`.
- We reviewed available record sets, fields and columns, and extracted data for processing and visualization.
- Through EDA, we filtered, normalized, and grouped the records using key attributes.
- Visualizations provided insight into distributions and relationships within second primary CRC survivor data.

Further analyses can be performed using more domain-specific criteria, leveraging the transparency and interoperability provided by Croissant schemas and the FAIR principles.